In [ ]:
!pip install tokenizers transformers nltk bert-score pandas torch scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import time
import math
import nltk
from bert_score import score
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast
from torch.utils.data import Dataset, DataLoader

nltk.download('punkt')
nltk.download('punkt_tab')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Advanced DL Environment Initialized. Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Advanced DL Environment Initialized. Using device: cuda


In [ ]:
print("Loading Training Data to train the Custom Tokenizer...")
train_sa_df = pd.read_csv('train_sa_10000.csv')
train_en_df = pd.read_csv('train_en_10000.csv')

all_training_texts = train_sa_df['Sentence_sa'].astype(str).tolist() + train_en_df['Sentence_en'].astype(str).tolist()

raw_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
raw_tokenizer.pre_tokenizer = Whitespace()

VOCAB_SIZE = 5000
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
    show_progress=True
)

print(f"Training Custom BPE Tokenizer (Target Vocab: {VOCAB_SIZE})...")
raw_tokenizer.train_from_iterator(all_training_texts, trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]"
)

tokenizer.save_pretrained("./custom_sanskrit_tokenizer")

vocab_size = len(tokenizer)
PAD_IDX = tokenizer.pad_token_id
BOS_IDX = tokenizer.bos_token_id
EOS_IDX = tokenizer.eos_token_id

print(f"Custom Tokenizer successfully trained! Final Vocabulary Size: {vocab_size}")

Loading Training Data to train the Custom Tokenizer...
Training Custom BPE Tokenizer (Target Vocab: 5000)...
Custom Tokenizer successfully trained! Final Vocabulary Size: 5000


In [ ]:
def load_translation_data(sa_path, en_path):
    df_sa = pd.read_csv(sa_path)
    df_en = pd.read_csv(en_path)
    return pd.merge(df_sa, df_en, on='Source_id', how='inner').sort_values('Source_id').reset_index(drop=True)

class TranslationDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sa_text = str(self.df.iloc[idx]['Sentence_sa'])
        en_text = str(self.df.iloc[idx]['Sentence_en'])

        src_text = f"[BOS] {sa_text} [EOS]"
        tgt_text = f"[BOS] {en_text} [EOS]"

        src_encoded = self.tokenizer(src_text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors="pt")
        tgt_encoded = self.tokenizer(tgt_text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors="pt")

        src_ids = src_encoded['input_ids'].squeeze()
        tgt_ids = tgt_encoded['input_ids'].squeeze()

        decoder_input = tgt_ids[:-1]
        expected_output = tgt_ids[1:]

        return src_ids, decoder_input, expected_output

print("Preparing DataLoaders...")
train_df = load_translation_data('train_sa_10000.csv', 'train_en_10000.csv')
dev_df = load_translation_data('dev_sa_1000.csv', 'dev_en_1000.csv')
test_df = load_translation_data('test_sa_1000.csv', 'test_en_1000.csv')

train_loader = DataLoader(TranslationDataset(train_df, tokenizer), batch_size=32, shuffle=True)
dev_loader = DataLoader(TranslationDataset(dev_df, tokenizer), batch_size=32, shuffle=False)

Preparing DataLoaders...


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class CustomSanskritTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=3, dim_feedforward=512, dropout=0.2):
        super(CustomSanskritTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

        self.fc_out.weight = self.embedding.weight

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src, tgt, tgt_mask=None, src_padding_mask=None, tgt_padding_mask=None):
        src_emb = self.pos_encoder(self.embedding(src))
        tgt_emb = self.pos_encoder(self.embedding(tgt))
        outs = self.transformer(
            src_emb, tgt_emb, tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask, tgt_key_padding_mask=tgt_padding_mask
        )
        return self.fc_out(outs)

In [ ]:
def calculate_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params

    print("\n--- Model Parameter Breakdown ---")
    print(f"Total Parameters:         {total_params:,}")
    print(f"Trainable Parameters:     {trainable_params:,}")
    print(f"Non-Trainable Parameters: {non_trainable_params:,}")
    print("---------------------------------\n")
    return total_params, trainable_params, non_trainable_params

print("Initializing the Upgraded Custom Transformer...")
model = CustomSanskritTransformer(vocab_size=vocab_size).to(device)

total_params, trainable_params, non_trainable_params = calculate_parameters(model)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

EPOCHS = 20
PATIENCE = 5
best_val_loss = float('inf')
epochs_no_improve = 0

print("Starting NMT Training...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for src, tgt_in, tgt_expected in train_loader:
        src, tgt_in, tgt_expected = src.to(device), tgt_in.to(device), tgt_expected.to(device)

        tgt_mask = model.generate_square_subsequent_mask(tgt_in.size(1)).to(device)
        src_padding_mask = (src == PAD_IDX).to(device)
        tgt_padding_mask = (tgt_in == PAD_IDX).to(device)

        optimizer.zero_grad()
        output = model(src, tgt_in, tgt_mask=tgt_mask, src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask)

        output = output.contiguous().view(-1, output.shape[-1])
        tgt_expected = tgt_expected.contiguous().view(-1)

        loss = criterion(output, tgt_expected)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, tgt_in, tgt_expected in dev_loader:
            src, tgt_in, tgt_expected = src.to(device), tgt_in.to(device), tgt_expected.to(device)
            tgt_mask = model.generate_square_subsequent_mask(tgt_in.size(1)).to(device)
            output = model(src, tgt_in, tgt_mask=tgt_mask, src_padding_mask=(src == PAD_IDX).to(device), tgt_padding_mask=(tgt_in == PAD_IDX).to(device))
            loss = criterion(output.contiguous().view(-1, output.shape[-1]), tgt_expected.contiguous().view(-1))
            val_loss += loss.item()

    avg_val_loss = val_loss / len(dev_loader)

    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_custom_nmt_model.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}!")
            break

model.load_state_dict(torch.load('best_custom_nmt_model.pth'))

Initializing the Upgraded Custom Transformer...

--- Model Parameter Breakdown ---
Total Parameters:         5,239,688
Trainable Parameters:     5,239,688
Non-Trainable Parameters: 0
---------------------------------

Starting NMT Training...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch [01/20] - Train Loss: 18.2028 | Val Loss: 9.3660
Epoch [02/20] - Train Loss: 8.9911 | Val Loss: 7.0428
Epoch [03/20] - Train Loss: 7.3974 | Val Loss: 6.5886
Epoch [04/20] - Train Loss: 6.7603 | Val Loss: 6.3475
Epoch [05/20] - Train Loss: 6.3921 | Val Loss: 6.2382
Epoch [06/20] - Train Loss: 6.1248 | Val Loss: 6.0337
Epoch [07/20] - Train Loss: 5.9112 | Val Loss: 5.9151
Epoch [08/20] - Train Loss: 5.7392 | Val Loss: 5.8430
Epoch [09/20] - Train Loss: 5.5863 | Val Loss: 5.7602
Epoch [10/20] - Train Loss: 5.4360 | Val Loss: 5.7042
Epoch [11/20] - Train Loss: 5.2958 | Val Loss: 5.6628
Epoch [12/20] - Train Loss: 5.1692 | Val Loss: 5.6153
Epoch [13/20] - Train Loss: 5.0439 | Val Loss: 5.5964
Epoch [14/20] - Train Loss: 4.9328 | Val Loss: 5.5567
Epoch [15/20] - Train Loss: 4.8297 | Val Loss: 5.5569
Epoch [16/20] - Train Loss: 4.7276 | Val Loss: 5.5386
Epoch [17/20] - Train Loss: 4.6411 | Val Loss: 5.5391
Epoch [18/20] - Train Loss: 4.5528 | Val Loss: 5.5202
Epoch [19/20] - Train Loss:

<All keys matched successfully>

In [ ]:
def translate_sentence(model, sentence, tokenizer, max_len=64):
    model.eval()
    src_text = f"[BOS] {sentence} [EOS]"
    src_encoded = tokenizer(src_text, return_tensors="pt", max_length=max_len, truncation=True)
    src = src_encoded['input_ids'].to(device)
    src_padding_mask = (src == PAD_IDX).to(device)

    tgt_ids = [BOS_IDX]

    for i in range(max_len):
        tgt_tensor = torch.LongTensor(tgt_ids).unsqueeze(0).to(device)
        tgt_mask = model.generate_square_subsequent_mask(tgt_tensor.size(1)).to(device)

        with torch.no_grad():
            output = model(src, tgt_tensor, tgt_mask=tgt_mask, src_padding_mask=src_padding_mask)

        next_token = output[0, -1, :].argmax().item()

        if next_token == EOS_IDX:
            break

        tgt_ids.append(next_token)

    return tokenizer.decode(tgt_ids[1:], skip_special_tokens=True)

print("Starting inference on the Test Set...")
start_time = time.time()

predicted_translations = []
gold_translations = test_df['Sentence_en'].tolist()

for idx, row in test_df.iterrows():
    pred_en = translate_sentence(model, row['Sentence_sa'], tokenizer)
    predicted_translations.append(pred_en)

end_time = time.time()
inference_time = end_time - start_time

print("\n--- OFFICIAL EVALUATION METRICS ---")
print(f"Total Parameters: {total_params:,}")
print(f"Inference Time: {inference_time:.2f} seconds")

# NLTK BLEU
bleu_scores = [nltk.translate.bleu_score.sentence_bleu([nltk.word_tokenize(g)], nltk.word_tokenize(p))
               for g, p in zip(gold_translations, predicted_translations)]
print(f"Average NLTK BLEU Score: {sum(bleu_scores) / len(bleu_scores):.4f}")

# BERTScore
print("Calculating BERTScore...")
P, R, F1 = score(predicted_translations, gold_translations, lang="en", verbose=False, rescale_with_baseline=True)
print(f"BERTScore (F1 Rescaled): {F1.mean().item():.4f}")

Starting inference on the Test Set...

--- OFFICIAL EVALUATION METRICS ---
Total Parameters: 5,239,688
Inference Time: 255.06 seconds


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

Average NLTK BLEU Score: 0.0033
Calculating BERTScore...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1 Rescaled): -0.1118


In [ ]:
print("Generating submission.csv...")
submission_df = pd.DataFrame({
    'Source_id': test_df['Source_id'],
    'Sentence_en': predicted_translations
})
submission_df.to_csv('submission.csv', index=False, encoding='utf-8')
print("Success! Custom BPE submission.csv is ready.")

Generating submission.csv...
Success! Custom BPE submission.csv is ready.
